<a href="https://colab.research.google.com/github/somaiah-ui/LangSmith-Project/blob/main/Smithbro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install -U langsmith langchain-google-genai gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.0/805.0 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 13.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


In [2]:
# ============================================================
# LANGSMITH AI CUSTOMER SUPPORT QUALITY LAB
# ============================================================

import os
import getpass
import re
import gradio as gr

from langsmith import traceable, Client
from langchain_google_genai import ChatGoogleGenerativeAI


# ============================================================
# 1. API KEYS
# ============================================================

print("🔑 Enter your API keys")
print("Your keys will stay hidden while typing.\n")

os.environ["LANGSMITH_API_KEY"] = getpass.getpass(
    "LangSmith API Key: "
)

os.environ["GOOGLE_API_KEY"] = getpass.getpass(
    "Gemini API Key: "
)


# ============================================================
# 2. LANGSMITH CONFIGURATION
# ============================================================

os.environ["LANGSMITH_TRACING"] = "true"

# Everything from this project will appear under this name
os.environ["LANGSMITH_PROJECT"] = "AI-Customer-Support-Quality-Lab"


# Connect to LangSmith
langsmith_client = Client()


# ============================================================
# 3. GEMINI MODELS
# ============================================================

# Main customer-support AI
support_llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.3
)

# Separate AI role that checks answer quality
checker_llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0
)


# ============================================================
# 4. FAKE COMPANY DATABASE
# ============================================================

COMPANY_KNOWLEDGE = {

    "pricing": """
Our company is called UniAI.

Plans:

Starter Plan:
$9/month
- Gemini access
- Limited daily messages
- Basic customer support

Pro Plan:
$19/month
- Gemini
- Claude
- ChatGPT
- Perplexity
- Higher message limits
- Priority support

Ultimate Plan:
$39/month
- All supported AI models
- Highest message limits
- Advanced research tools
- Priority customer support
""",

    "refund": """
Refund Policy:

Customers can request a refund within 7 days of purchase.

Refunds may be rejected if the customer has used a very large
amount of the service.

Approved refunds normally return to the original payment method.

Processing time depends on the payment provider.
""",

    "technical": """
Technical Support:

If a chatbot is unavailable:

1. Refresh the application.
2. Sign out and sign back in.
3. Check whether another AI model works.
4. Try again after a short period.

If the issue continues, contact technical support.

Never ask a customer to provide their password.
""",

    "account": """
Account Information:

Users can change their email, password and profile information
from Account Settings.

Passwords cannot be viewed by support staff.

Users should never send passwords to customer support.

Account deletion can be requested from Account Settings.
""",

    "general": """
UniAI provides access to several AI assistants through a single
subscription.

Depending on the subscription plan, users may access models
from providers such as Google, OpenAI, Anthropic and Perplexity.
"""
}


# ============================================================
# HELPER FUNCTION
# ============================================================

def text_from_response(response):

    content = response.content

    if isinstance(content, str):
        return content

    if isinstance(content, list):

        pieces = []

        for block in content:

            if isinstance(block, dict):

                if "text" in block:
                    pieces.append(block["text"])

            else:
                pieces.append(str(block))

        return "\n".join(pieces)

    return str(content)


# ============================================================
# 5. CLASSIFIER
# ============================================================

@traceable(
    name="01_Classify_Customer_Request",
    run_type="chain"
)
def classify_question(question):

    prompt = f"""
You are a customer-support request classifier.

Classify the customer's question into exactly ONE category:

pricing
refund
technical
account
general

Customer question:
{question}

Return ONLY the category.
"""

    response = support_llm.invoke(prompt)

    category = text_from_response(response).lower().strip()

    allowed = [
        "pricing",
        "refund",
        "technical",
        "account",
        "general"
    ]

    for item in allowed:

        if item in category:
            return item

    return "general"


# ============================================================
# 6. KNOWLEDGE RETRIEVAL
# ============================================================

@traceable(
    name="02_Retrieve_Company_Knowledge",
    run_type="retriever"
)
def retrieve_information(category):

    return COMPANY_KNOWLEDGE.get(
        category,
        COMPANY_KNOWLEDGE["general"]
    )


# ============================================================
# 7. ANSWER GENERATOR
# ============================================================

@traceable(
    name="03_Generate_Support_Answer",
    run_type="chain"
)
def generate_answer(question, category, knowledge):

    prompt = f"""
You are a helpful customer-support assistant for UniAI.

Customer category:
{category}

Company information:

{knowledge}

Customer question:

{question}

Rules:

1. Answer using the company information.
2. Do not invent policies.
3. Keep the answer easy to understand.
4. Be friendly.
5. Keep the response reasonably short.
6. If the information isn't available, clearly say so.

Answer:
"""

    response = support_llm.invoke(prompt)

    return text_from_response(response)


# ============================================================
# 8. QUALITY CHECKER
# ============================================================

@traceable(
    name="04_Check_Answer_Quality",
    run_type="chain"
)
def quality_check(question, knowledge, answer):

    prompt = f"""
You are a quality-control AI.

Check whether this customer-support answer is:

- Correct
- Based on the supplied company information
- Clear
- Helpful
- Not hallucinated

Company information:

{knowledge}

Customer question:

{question}

AI answer:

{answer}

Respond in EXACTLY this structure:

DECISION: PASS or REVISE
SCORE: number from 1 to 10
REASON: short explanation
"""

    response = checker_llm.invoke(prompt)

    result = text_from_response(response)

    return result


# ============================================================
# 9. REWRITE BAD ANSWERS
# ============================================================

@traceable(
    name="05_Improve_Weak_Answer",
    run_type="chain"
)
def improve_answer(question, knowledge, old_answer, feedback):

    prompt = f"""
You are a senior customer-support assistant.

The previous AI response failed quality checking.

Customer question:

{question}

Company information:

{knowledge}

Previous answer:

{old_answer}

Quality checker feedback:

{feedback}

Rewrite the answer.

Rules:

- Fix the problems identified by the checker.
- Use only the supplied company information.
- Do not invent facts.
- Be friendly.
- Be concise.
- Give only the final customer-facing answer.
"""

    response = support_llm.invoke(prompt)

    return text_from_response(response)


# ============================================================
# 10. COMPLETE SUPPORT PIPELINE
# ============================================================

@traceable(
    name="UniAI_Customer_Support_Pipeline",
    run_type="chain"
)
def customer_support(question):

    # --------------------------------------
    # Step 1: classify
    # --------------------------------------

    category = classify_question(question)


    # --------------------------------------
    # Step 2: retrieve information
    # --------------------------------------

    knowledge = retrieve_information(category)


    # --------------------------------------
    # Step 3: generate answer
    # --------------------------------------

    answer = generate_answer(
        question,
        category,
        knowledge
    )


    # --------------------------------------
    # Step 4: quality check
    # --------------------------------------

    evaluation = quality_check(
        question,
        knowledge,
        answer
    )


    # --------------------------------------
    # Step 5: rewrite if needed
    # --------------------------------------

    if "REVISE" in evaluation.upper():

        final_answer = improve_answer(
            question,
            knowledge,
            answer,
            evaluation
        )

        status = "Answer was automatically improved"

    else:

        final_answer = answer

        status = "Answer passed quality checking"


    return {
        "answer": final_answer,
        "category": category,
        "evaluation": evaluation,
        "status": status
    }


# ============================================================
# 11. GRADIO FUNCTION
# ============================================================

def chat(question):

    if not question.strip():
        return "Please enter a question.", "", "", ""

    try:

        result = customer_support(question)

        # Push pending traces to LangSmith
        try:
            langsmith_client.flush()
        except:
            pass

        return (
            result["answer"],
            result["category"],
            result["evaluation"],
            result["status"]
        )

    except Exception as e:

        return (
            f"❌ Error: {str(e)}",
            "Error",
            "Could not evaluate",
            "Pipeline failed"
        )


# ============================================================
# 12. GRADIO UI
# ============================================================

with gr.Blocks() as demo:

    gr.Markdown(
        """
# 🔬 LangSmith AI Customer Support Quality Lab

Ask a customer-support question.

Every step of the AI pipeline is automatically traced
inside **LangSmith**.
"""
    )

    question = gr.Textbox(
        label="Customer Question",
        placeholder="Example: Can I get a refund?"
    )

    submit = gr.Button("Ask Support AI")


    answer = gr.Textbox(
        label="Final Customer Answer",
        lines=6
    )


    with gr.Row():

        category = gr.Textbox(
            label="Detected Category"
        )

        status = gr.Textbox(
            label="Quality Status"
        )


    evaluation = gr.Textbox(
        label="AI Quality Checker",
        lines=4
    )


    submit.click(
        chat,
        inputs=question,
        outputs=[
            answer,
            category,
            evaluation,
            status
        ]
    )


demo.launch(debug=False)

🔑 Enter your API keys
Your keys will stay hidden while typing.

LangSmith API Key: ··········
Gemini API Key: ··········
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c25c06fcc31e383bf1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
